In [3]:
import numpy as np
import pandas as pd

!pip install CTGAN

from ctgan import CTGAN

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.4 MB/s eta 0:00:00


In [1]:
from google.colab import files
uploaded = files.upload()

Saving X_domain_processed.csv to X_domain_processed.csv
Saving X_raw_processed.csv to X_raw_processed.csv


In [4]:
# =========================================================
# LOAD DATA
# =========================================================
df_raw = pd.read_csv("X_raw_processed.csv")
df_domain = pd.read_csv("X_domain_processed.csv")

target_col = "Heart Disease Status"

X_domain = df_domain.drop(columns=[target_col]).values
y_domain = df_domain[target_col].values

In [5]:
# =========================================================
# ENTROPY
# =========================================================
def entropy(p):
    eps = 1e-10
    return -(p * np.log(p + eps) + (1 - p) * np.log(1 - p + eps))

In [6]:
# =========================================================
# CORAL (REAL DOMAIN ALIGNMENT)
# =========================================================
def coral(source, target):

    source = np.asarray(source)
    target = np.asarray(target)

    src_mean = source.mean(axis=0)
    tgt_mean = target.mean(axis=0)

    source_c = source - src_mean
    target_c = target - tgt_mean

    cov_src = np.cov(source_c, rowvar=False) + np.eye(source.shape[1]) * 1e-6
    cov_tgt = np.cov(target_c, rowvar=False) + np.eye(target.shape[1]) * 1e-6

    U_s, S_s, _ = np.linalg.svd(cov_src)
    U_t, S_t, _ = np.linalg.svd(cov_tgt)

    whitening = U_s @ np.diag(1.0 / np.sqrt(S_s)) @ U_s.T
    coloring  = U_t @ np.diag(np.sqrt(S_t)) @ U_t.T

    aligned = source_c @ whitening @ coloring + tgt_mean

    return aligned

In [7]:
# =========================================================
# SYNTHETIC DATA (CTGAN)
# =========================================================
def generate_synthetic(df, percent):

    n = int(len(df) * percent)

    major = df[df[target_col] == 0]
    minor = df[df[target_col] == 1]

    ctgan_m = CTGAN(epochs=200)
    ctgan_n = CTGAN(epochs=200)

    ctgan_m.fit(major)
    ctgan_n.fit(minor)

    ratio_m = len(major) / len(df)
    n_m = int(n * ratio_m)
    n_n = n - n_m

    synth_m = ctgan_m.sample(n_m)
    synth_n = ctgan_n.sample(n_n)

    synth = pd.concat([synth_m, synth_n], ignore_index=True)

    return synth.sample(frac=1, random_state=42).reset_index(drop=True)

In [8]:
# =========================================================
# CLASS WEIGHT HELPER
# =========================================================
def get_class_weight(y):
    pos = np.sum(y == 1)
    neg = np.sum(y == 0)

    return {
        0: 1.0,
        1: neg / (pos + 1e-6)
    }

In [9]:
# =========================================================
# MODELS (WITH CLASS WEIGHTS)
# =========================================================
def build_models(y_train):

    cw = get_class_weight(y_train)

    e0_models = [
        LogisticRegression(solver='liblinear', class_weight=cw, random_state=42),
        XGBClassifier(eval_metric='logloss', scale_pos_weight=cw[1], random_state=42),
        ExtraTreesClassifier(n_estimators=200, class_weight=cw, random_state=42)
    ]

    e1_models = [
        RandomForestClassifier(n_estimators=200, class_weight=cw, random_state=42),
        LogisticRegression(solver='liblinear', class_weight=cw, random_state=42)
    ]

    e2_model = XGBClassifier(
        eval_metric='logloss',
        scale_pos_weight=cw[1],
        random_state=42
    )

    return e0_models, e1_models, e2_model

In [10]:

# =========================================================
# OOF STACKING
# =========================================================
def oof(models, X, y, X_test, k=5):

    kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    oof_tr = np.zeros((len(X), len(models)))
    oof_te = np.zeros((len(X_test), len(models)))

    for i, model in enumerate(models):

        test_folds = np.zeros((len(X_test), k))

        for f, (tr, val) in enumerate(kf.split(X, y)):

            X_tr, X_val = X[tr], X[val]
            y_tr = y[tr]

            model.fit(X_tr, y_tr)

            oof_tr[val, i] = model.predict_proba(X_val)[:, 1]
            test_folds[:, f] = model.predict_proba(X_test)[:, 1]

        oof_te[:, i] = test_folds.mean(axis=1)

    return oof_tr, oof_te

In [ ]:
# =========================================================
# THRESHOLD TUNING
# =========================================================
def best_threshold(y, preds):

    best_t, best_f1 = 0.5, 0

    for t in np.linspace(0.05, 0.95, 50):

        p = (preds >= t).astype(int)
        f1 = f1_score(y, p)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1

In [11]:
# =========================================================
# ARCHITECTURE: ORIGINAL
# =========================================================
def run_original(X_train, y_train, X_test, models):

    e0_m, e1_m, e2_m = models

    e0_tr, e0_te = oof(e0_m, X_train, y_train, X_test)
    e1_tr, e1_te = oof(e1_m, e0_tr, y_train, e0_te)
    e2_tr, e2_te = oof([e2_m], e1_tr, y_train, e1_te)

    return e0_tr.mean(1), e1_tr.mean(1), e2_tr[:,0], \
           e0_te.mean(1), e1_te.mean(1), e2_te[:,0]

In [12]:
# =========================================================
# ARCHITECTURE: RESIDUAL
# =========================================================
def run_residual(X_train, y_train, X_test, models):

    e0_m, e1_m, e2_m = models

    e0_tr, e0_te = oof(e0_m, X_train, y_train, X_test)

    X1_tr = np.hstack([X_train, e0_tr])
    X1_te = np.hstack([X_test, e0_te])

    e1_tr, e1_te = oof(e1_m, X1_tr, y_train, X1_te)

    X2_tr = np.hstack([X_train, e1_tr])
    X2_te = np.hstack([X_test, e1_te])

    e2_tr, e2_te = oof([e2_m], X2_tr, y_train, X2_te)

    return e0_tr.mean(1), e1_tr.mean(1), e2_tr[:,0], \
           e0_te.mean(1), e1_te.mean(1), e2_te[:,0]

In [13]:
# =========================================================
# ARCHITECTURE: ENTROPY
# =========================================================
def run_entropy(X_train, y_train, X_test, models):

    e0_m, e1_m, e2_m = models

    e0_tr, e0_te = oof(e0_m, X_train, y_train, X_test)

    e0_ent_tr = entropy(e0_tr)
    e0_ent_te = entropy(e0_te)

    X1_tr = np.hstack([X_train, e0_tr, e0_ent_tr])
    X1_te = np.hstack([X_test, e0_te, e0_ent_te])

    e1_tr, e1_te = oof(e1_m, X1_tr, y_train, X1_te)

    e1_ent_tr = entropy(e1_tr)
    e1_ent_te = entropy(e1_te)

    X2_tr = np.hstack([X_train, e1_tr, e1_ent_tr])
    X2_te = np.hstack([X_test, e1_te, e1_ent_te])

    e2_tr, e2_te = oof([e2_m], X2_tr, y_train, X2_te)

    return e0_tr.mean(1), e1_tr.mean(1), e2_tr[:,0], \
           e0_te.mean(1), e1_te.mean(1), e2_te[:,0]

In [14]:
# =========================================================
# EVALUATION
# =========================================================
def evaluate(name, y_train, y_test,
             e0_tr, e1_tr, e2_tr,
             e0_te, e1_te, e2_te):

    print(f"\n================ {name} ================")

    for lvl, tr, te in zip(
        ["E0","E1","E2"],
        [e0_tr, e1_tr, e2_tr],
        [e0_te, e1_te, e2_te]
    ):

        t, f1 = best_threshold(y_train, tr)
        preds = (te >= t).astype(int)

        print(f"\n--- {lvl} ---")
        print("ROC-AUC:", roc_auc_score(y_test, te))
        print("Best Threshold:", t)
        print("F1:", f1)
        print("Accuracy:", accuracy_score(y_test, preds))
        print(classification_report(y_test, preds))

In [ ]:
# =========================================================
# EXPERIMENT LOOP
# =========================================================
ratios = [0, 0.5]
results = {}

for r in ratios:

    print("\n#################################################")
    print("SYNTHETIC RATIO:", r)
    print("#################################################")

    df = df_raw.copy() if r == 0 else pd.concat(
        [df_raw, generate_synthetic(df_raw, r)],
        ignore_index=True
    )

    X = df.drop(columns=[target_col]).values
    y = df[target_col].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    X_domain_shift = X_domain

    models = build_models(y_train)

    for name, fn in [
        ("ORIGINAL", run_original),
        ("RESIDUAL", run_residual),
        ("ENTROPY", run_entropy)
    ]:

        # ---------------- IN DOMAIN ----------------
        e0_tr, e1_tr, e2_tr, e0_te, e1_te, e2_te = fn(X_train, y_train, X_test, models)

        evaluate(name+" IN", y_train, y_test,
                 e0_tr, e1_tr, e2_tr,
                 e0_te, e1_te, e2_te)

        # ---------------- DOMAIN SHIFT + CORAL ----------------
        X_train_coral = coral(X_train, X_domain_shift)

        e0_tr, e1_tr, e2_tr, e0_te, e1_te, e2_te = fn(X_train_coral, y_train, X_domain_shift, models)

        evaluate(name+" CORAL SHIFT", y_train, y_domain,
                 e0_tr, e1_tr, e2_tr,
                 e0_te, e1_te, e2_te)

print("\nALL EXPERIMENTS COMPLETE")